# Task
Generate Python code to perform transfer learning for vehicle image classification using the dataset located at "/content/drive/MyDrive/Vehicles". The code should include data loading, preprocessing, model building using a pre-trained CNN, training, and evaluation, and finally output the model's accuracy.

## Load and preprocess data

### Subtask:
Load the images from the specified directory, preprocess them (e.g., resize, normalize), and split the data into training and validation sets.


**Reasoning**:
Load and preprocess the images using ImageDataGenerator and split the data into training and validation sets.



In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Define the path to the dataset
dataset_path = "/content/drive/MyDrive/Vehicles"

# Define image size and batch size
img_height = 224
img_width = 224
batch_size = 32

# Create an ImageDataGenerator for training with data augmentation and validation split
train_datagen = ImageDataGenerator(
    rescale=1./255,  # Normalize pixel values to [0, 1]
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2) # 20% of the data will be used for validation

# Load and preprocess training data
train_generator = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical', # Use 'categorical' for multi-class classification
    subset='training') # Specify training subset

# Load and preprocess validation data
validation_generator = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation') # Specify validation subset

print("Data loading and preprocessing complete.")
print("Number of training batches:", len(train_generator))
print("Number of validation batches:", len(validation_generator))

Found 4470 images belonging to 7 classes.
Found 1117 images belonging to 7 classes.
Data loading and preprocessing complete.
Number of training batches: 140
Number of validation batches: 35


## Load a pre-trained model

### Subtask:
Load a pre-trained convolutional neural network (CNN) model (e.g., ResNet50, VGG16) without the top classification layer.


**Reasoning**:
Import the ResNet50 model and load it without the top classification layer, using pre-trained weights.



In [2]:
from tensorflow.keras.applications import ResNet50

# Load the ResNet50 model with pre-trained weights and exclude the top layer
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(img_height, img_width, 3))

print("ResNet50 base model loaded successfully.")

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
ResNet50 base model loaded successfully.


## Build the new model

### Subtask:
Add new layers to the pre-trained model for classification, matching the number of vehicle classes in the dataset.


**Reasoning**:
Add new layers to the pre-trained model for classification, matching the number of vehicle classes in the dataset.



In [3]:
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model

# Define the number of classes
num_classes = train_generator.num_classes

# Add new layers for classification
x = base_model.output
x = GlobalAveragePooling2D()(x)
predictions = Dense(num_classes, activation='softmax')(x)

# Create the new model
model = Model(inputs=base_model.input, outputs=predictions)

print("New layers added to the base model.")
print("Number of classes:", num_classes)

New layers added to the base model.
Number of classes: 7


## Compile the model

### Subtask:
Compile the model with an appropriate optimizer, loss function, and metrics.


**Reasoning**:
Compile the model with the Adam optimizer, categorical crossentropy loss, and accuracy as a metric.



In [4]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy

# Compile the model
model.compile(optimizer=Adam(),
              loss=CategoricalCrossentropy(),
              metrics=['accuracy'])

print("Model compilation complete.")

Model compilation complete.


## Train the model

### Subtask:
Train the new model on the training data, using the validation data for monitoring performance.


**Reasoning**:
Train the model using the specified generators and number of epochs.



In [5]:
# Train the model
history = model.fit(
    train_generator,
    epochs=10,
    validation_data=validation_generator)

print("Model training complete.")

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
  1/140 ━━━━━━━━━━━━━━━━━━━━ 3:04:26 80s/step - accuracy: 0.0938 - loss: 2.5055

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


140/140 ━━━━━━━━━━━━━━━━━━━━ 3943s 28s/step - accuracy: 0.7223 - loss: 1.0157 - val_accuracy: 0.1432 - val_loss: 31.9234
Epoch 2/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 3654s 26s/step - accuracy: 0.9144 - loss: 0.2678 - val_accuracy: 0.1432 - val_loss: 2.4625
Epoch 3/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 3723s 27s/step - accuracy: 0.9360 - loss: 0.1951 - val_accuracy: 0.1432 - val_loss: 3.1066
Epoch 4/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 3729s 27s/step - accuracy: 0.9488 - loss: 0.1529 - val_accuracy: 0.1567 - val_loss: 10.8064
Epoch 5/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 3715s 27s/step - accuracy: 0.9642 - loss: 0.1136 - val_accuracy: 0.4342 - val_loss: 1.5615
Epoch 6/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 3697s 26s/step - accuracy: 0.9614 - loss: 0.1247 - val_accuracy: 0.7610 - val_loss: 0.7782
Epoch 7/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 3665s 26s/step - accuracy: 0.9623 - loss: 0.1100 - val_accuracy: 0.8550 - val_loss: 0.5030
Epoch 8/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 3631s 26s/step - accuracy: 0.9579 - loss: 0.1119 - v

## Evaluate the model

### Subtask:
Evaluate the trained model on the validation data to calculate the accuracy.


**Reasoning**:
Evaluate the trained model on the validation data to calculate the accuracy and print the result.



In [6]:
# Evaluate the model on the validation data
evaluation_results = model.evaluate(validation_generator)

# Extract and print the accuracy
accuracy = evaluation_results[1]
print(f"Validation Accuracy: {accuracy:.4f}")

35/35 ━━━━━━━━━━━━━━━━━━━━ 199s 6s/step - accuracy: 0.8294 - loss: 0.5990
Validation Accuracy: 0.8299


## Summary:

### Data Analysis Key Findings

*   The dataset was successfully loaded and preprocessed, with images resized to 224x224 and normalized.
*   The dataset was split into training (4470 images) and validation (1117 images) sets across 7 classes.
*   A pre-trained ResNet50 model, excluding the top classification layer, was loaded using 'imagenet' weights.
*   New layers, including `GlobalAveragePooling2D` and a `Dense` layer with softmax activation matching the 7 classes, were added to the base model.
*   The model was compiled using the Adam optimizer, Categorical Crossentropy loss, and accuracy as the evaluation metric.
*   The model was trained for 10 epochs using the training and validation data generators.
*   The final validation accuracy of the trained model is approximately 0.8299 (82.99%).

### Insights or Next Steps

*   The model achieved a respectable accuracy of over 80% on the validation set, indicating successful transfer learning for vehicle classification.
*   Further steps could involve fine-tuning the base ResNet50 layers (unfreezing some initial layers) or experimenting with different pre-trained models and hyperparameters to potentially improve accuracy.
